In [2]:
import pandas as pd
import numpy as np

pool = pd.read_parquet("data/pool_splits.parquet")

def write_fasta(frame, path, seq_col="protein_stripped"):
    with open(path, "w") as f:
        for idx, seq in zip(frame.index, frame[seq_col]):
            f.write(f">{idx}\n{seq}\n")

for split in ["split_random", "split_cluster"]:
    name = split.replace("split_", "")
    write_fasta(pool[pool[split] == "train"], f"data/{name}_train.fasta")
    write_fasta(pool[pool[split] == "test"], f"data/{name}_test.fasta")

In [3]:
cols = ["query", "target", "fident", "alnlen", "qcov", "tcov", "evalue", "bits"]

def best_hits(path):
    hits = pd.read_csv(path, sep="\t", header=None, names=cols)
    hits = hits.sort_values("bits", ascending=False)
    return hits.drop_duplicates(subset="query", keep="first").set_index("query")

best_clu = best_hits("data/search/cluster.m8")
best_rand = best_hits("data/search/random.m8")

In [4]:
clu_test = pool.index[pool["split_cluster"] == "test"]
rand_test = pool.index[pool["split_random"] == "test"]

print(len(best_clu), len(clu_test), len(best_clu) / len(clu_test))
print(len(best_rand), len(rand_test), len(best_rand) / len(rand_test))

9998 13885 0.7200576161325171
11555 13885 0.8321930140439323


In [5]:
print(best_clu["fident"].describe())
print(best_rand["fident"].describe())

count    9998.000000
mean        0.402481
std         0.179704
min         0.175000
25%         0.289250
50%         0.339000
75%         0.441000
max         1.000000
Name: fident, dtype: float64
count    11555.000000
mean         0.551699
std          0.234557
min          0.187000
25%          0.359000
50%          0.485000
75%          0.715000
max          1.000000
Name: fident, dtype: float64


In [6]:
high = best_clu[best_clu["fident"] > 0.9]
print(len(high))
print(high[["fident", "alnlen", "qcov", "tcov"]].describe())

445
           fident      alnlen        qcov        tcov
count  445.000000  445.000000  445.000000  445.000000
mean     0.988611  196.665169    0.721613    0.690045
std      0.022796  117.179514    0.258687    0.291084
min      0.902000   18.000000    0.032000    0.052000
25%      0.990000  112.000000    0.517000    0.439000
50%      1.000000  173.000000    0.742000    0.738000
75%      1.000000  246.000000    0.991000    0.990000
max      1.000000  910.000000    1.000000    1.000000


In [7]:
print(best_clu[["fident", "qcov", "tcov", "alnlen"]].describe())
print(best_rand[["fident", "qcov", "tcov", "alnlen"]].describe())

            fident         qcov         tcov       alnlen
count  9998.000000  9998.000000  9998.000000  9998.000000
mean      0.402481     0.755396     0.721353   230.611722
std       0.179704     0.215534     0.246420   119.018803
min       0.175000     0.032000     0.032000    18.000000
25%       0.289250     0.611000     0.548000   141.000000
50%       0.339000     0.814000     0.790000   216.000000
75%       0.441000     0.939000     0.936000   294.000000
max       1.000000     1.000000     1.000000  1342.000000
             fident          qcov          tcov        alnlen
count  11555.000000  11555.000000  11555.000000  11555.000000
mean       0.551699      0.879503      0.847584    266.108178
std        0.234557      0.176721      0.215021    137.630753
min        0.187000      0.057000      0.030000     18.000000
25%        0.359000      0.856000      0.811000    166.000000
50%        0.485000      0.956000      0.947000    247.000000
75%        0.715000      0.989000      0.987

In [8]:
leaky = best_clu[
    (best_clu["fident"] >= 0.3)
    & (best_clu["qcov"] >= 0.8)
    & (best_clu["tcov"] >= 0.8)
]
print(len(leaky), len(leaky) / len(clu_test))

1229 0.08851278357940223


In [9]:
leaky_r = best_rand[
    (best_rand["fident"] >= 0.3)
    & (best_rand["qcov"] >= 0.8)
    & (best_rand["tcov"] >= 0.8)
]
print(len(leaky_r), len(leaky_r) / len(rand_test))

7375 0.5311487216420597


Under a random split, 53% of test proteins had a close homolog (≥30% identity, ≥80% coverage both ways) in training. Cluster-based splitting reduced this to 8.9%; the residual arises because MMseqs2 clustering is greedy and does not perform exhaustive pairwise comparison, so some homologous pairs are assigned to different clusters.

In [10]:
labels = pool["label"]
clu_test = pool.index[pool["split_cluster"] == "test"]
rand_test = pool.index[pool["split_random"] == "test"]

In [11]:
def lookup_predict(best, test_idx):
    matched = best["target"].map(labels)
    preds = matched.reindex(test_idx)
    fallback_rate = preds.isna().mean()
    return preds.fillna(0).astype(int), fallback_rate

In [12]:
pred_clu, fb_clu = lookup_predict(best_clu, clu_test)
pred_rand, fb_rand = lookup_predict(best_rand, rand_test)
print(fb_clu, fb_rand)

0.2799423838674829 0.1678069859560677


In [13]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    f1_score, matthews_corrcoef,
)

def score(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
    }

print("clustered:", score(labels[clu_test], pred_clu))
print("random:   ", score(labels[rand_test], pred_rand))

clustered: {'accuracy': 0.6115952466690674, 'balanced_acc': 0.5840053816480044, 'f1': 0.47267038232130637, 'mcc': 0.17772698808928197}
random:    {'accuracy': 0.6140439323010443, 'balanced_acc': 0.5957749903746783, 'f1': 0.512064099062187, 'mcc': 0.19525329649755926}


In [14]:
matched_clu = best_clu.index.intersection(clu_test)
matched_rand = best_rand.index.intersection(rand_test)

print("clustered, matched only:",
      score(labels[matched_clu], pred_clu[matched_clu]))
print("random, matched only:   ",
      score(labels[matched_rand], pred_rand[matched_rand]))

clustered, matched only: {'accuracy': 0.6138227645529106, 'balanced_acc': 0.6078101324406744, 'f1': 0.5559516963772283, 'mcc': 0.21465572496597635}
random, matched only:    {'accuracy': 0.6082215491129381, 'balanced_acc': 0.6030779467085559, 'f1': 0.554034085311792, 'mcc': 0.20516774392867362}


In [16]:
ex = best_rand.loc[close[:5]]
for q, row in ex.iterrows():
    t = row["target"]
    print("query", q, "label", labels[q], "| target", t, "label", labels[t],
          "| fident", row["fident"])
    print("  q:", pool.loc[q, "protein_stripped"][:60])
    print("  t:", pool.loc[t, "protein_stripped"][:60])

query 15610 label 0 | target 27560.0 label 1 | fident 0.999
  q: MLKFRTVHGGLRLLGIRRTSTAPAASPNVRRLEYKPIKKVMVANRGEIAIRVFRACTELG
  t: MLKFRTVHGGLRLLGIRRTSTAPAASPNVRRLEYKPIKKVMVANRGEIAIRVFRACTELG
query 47508 label 0 | target 5286.0 label 1 | fident 1.0
  q: MTMHREPGGERYYYTWAWFEGPDDAAWRVTGHHTDSGEQYRLDWNLAERSLCVTDSLGRT
  t: MTMHREPGGERYYYTWAWFEGPDDAAWRVTGHHTDSGEQYRLDWNLAERSLCVTDSLGRT
query 3487 label 1 | target 47741.0 label 0 | fident 0.998
  q: MAELNFKAIEEKWQKRWLEAKIFEPNIRDKPKEKKFYITVAFPYLSGHLHVGHARTYTIP
  t: MSLAELNFKAIEEKWQKRWLEAKIFEPNIRDKPKEKKFYITVAFPYLSGHLHVGHARTYT
query 9584 label 0 | target 59234.0 label 1 | fident 0.998
  q: VADRLIVKGAREHNLRSVDLDLPRDALIVFTGLSGSGKSSLAFDTIFAEGQRRYVESLSA
  t: MADRLIVKGAREHNLRSVDLDLPRDALIVFTGLSGSGKSSLAFDTIFAEGQRRYVESLSA
query 28843 label 0 | target 10871.0 label 1 | fident 0.968
  q: MSLQHMYRCARVQGLDTSEGLLLFGKEHFYVIDGFTMTATREIRDIETLPPNMHEPIIPR
  t: IDNLCLTPASKQTPLIDLLLEASPERSTRTQQKEFQTYILDSVMDHLLAADVLLGEDASL


In [17]:
print(best_rand["target"].dtype, pool.index.dtype)
print(best_rand.index.dtype)

int64 int64
int64


In [18]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(labels[close], pred_rand[close]))

[[ 66 696]
 [647  82]]


In [19]:
b = best_rand.loc[matched_rand].copy()
b["q_label"] = labels[b.index].values
b["t_label"] = b["target"].astype(int).map(labels).values
b["agree"] = (b["q_label"] == b["t_label"])

bands = [0, 0.3, 0.4, 0.5, 0.7, 0.9, 1.01]
b.groupby(pd.cut(b["fident"], bands), observed=True)["agree"].agg(["mean", "size"])

,mean,size
fident,,
"(0.0, 0.3]",0.617156,1259
"(0.3, 0.4]",0.652205,2766
"(0.4, 0.5]",0.681416,2034
"(0.5, 0.7]",0.713998,2479
"(0.7, 0.9]",0.749017,1526
"(0.9, 1.01]",0.099262,1491
